# RAIL lung-nodule counting — Colab setup

Before running anything:
1. **Runtime > Change runtime type > GPU** (T4 is fine).
2. Upload `lidc_idri_p1-20.zip` to your Google Drive. This pipeline only ever uses patients 1-20 (every script defaults to `--start 1 --end 20`), so you don't need the full 99-patient/11GB `LDIC-IDRI-subset` — just those 20 patients + `annotations.csv`, zipped to ~1.1GB. (If you need more patients later, re-zip a wider range the same way.)
3. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) holding a Hugging Face access token that has accepted the MedGemma license at https://huggingface.co/google/medgemma-1.5-4b-it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/freya-gul/rail.git
%cd rail

Point this at wherever you uploaded `lidc_idri_p1-20.zip` in Drive. It unzips into `datasets/LDIC-IDRI-subset/` inside the cloned repo — that's the path every script's `DICOM_ROOT` already expects, so nothing else needs to change:

In [ ]:
ZIP_PATH = "/content/drive/MyDrive/lidc_idri_p1-20.zip"  # <-- update to your actual upload path
DATA_DIR = "datasets/LDIC-IDRI-subset"  # relative to the repo root (we've already %cd'd into rail)

import pathlib
assert pathlib.Path(ZIP_PATH).exists(), f"{ZIP_PATH} not found — check the path/upload"

In [ ]:
!mkdir -p {DATA_DIR}
!unzip -q {ZIP_PATH} -d {DATA_DIR}
!ls {DATA_DIR}

In [ ]:
# This notebook only runs the MedGemma counting baseline (8_evaluate_medgemma_counting.py),
# which doesn't touch pylidc/monai/simpleitk at all - only pydicom + transformers are needed
# beyond what Colab already has. See requirements-colab.txt for the full install if you also
# want to run the MONAI detector (7_evaluate_detection.py) or regenerate ground truth (6_*.py).
!pip install -q pydicom "transformers>=5.12.1" "huggingface_hub>=1.21.0"

Sanity check: GPU visible to torch, and the detector/MedGemma scripts will now pick it (they already default to `cuda` > `mps` > `cpu`):

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Run MedGemma counting

Compares MedGemma 1.5's whole-volume nodule count against two independent ground truths, already committed in the repo for patients 1-20:
- `gt_count_pylidc` — every pylidc consensus annotation cluster (`ground_truth_annotations.json`)
- `gt_count_luna16` — the actual external LUNA16 challenge `annotations.csv`, joined by SeriesInstanceUID (a separately-collected nodule list, not derived from pylidc)

In [ ]:
# MedGemma-only whole-volume counting baseline vs. ground truth (GPU)
!python image_download/8_evaluate_medgemma_counting.py --start 1 --end 20

Results land at `image_download/medgemma_counting_comparison.csv`, with per-patient `predicted_count`, `gt_count_pylidc`, `gt_count_luna16`, and both error columns, plus MAE/bias vs. each ground truth printed above. The run caches each patient's MedGemma response to `medgemma_count_cache/<patient>.json`, so it's safe to interrupt and rerun — already-cached patients are skipped.